# Real-World Ingestion Patterns Examples

This notebook collects practical Databricks ingestion examples for common source types.

The flow covers:

- Kafka event ingestion
- IoT sensor telemetry ingestion
- Auto Loader file landing
- API payload landing into bronze

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType, TimestampType

## Kafka ingestion pattern

A common pattern is to read raw messages from Kafka, parse the JSON payload, and land the raw-plus-metadata stream into a bronze Delta table.

In [ ]:
kafka_bootstrap_servers = "broker1:9092,broker2:9092"
kafka_topic = "orders.events"
kafka_checkpoint_path = "/Volumes/main/demo/checkpoints/orders_kafka"
kafka_target_table = "main.demo.orders_kafka_bronze"

orders_event_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("amount", DoubleType(), True)
])

In [ ]:
kafka_bronze_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers)
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "latest")
    .load()
    .select(
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_ingest_ts"),
        F.col("key").cast("string").alias("message_key"),
        F.col("value").cast("string").alias("raw_payload")
    )
    .withColumn("payload", F.from_json("raw_payload", orders_event_schema))
    .select(
        "topic",
        "partition",
        "offset",
        "kafka_ingest_ts",
        "message_key",
        "raw_payload",
        F.col("payload.order_id").alias("order_id"),
        F.col("payload.customer_id").alias("customer_id"),
        F.col("payload.event_type").alias("event_type"),
        F.to_timestamp(F.col("payload.event_ts")).alias("event_ts"),
        F.col("payload.amount").alias("amount"),
        F.current_timestamp().alias("bronze_ingest_ts")
    )
)

kafka_bronze_stream_df

## IoT sensor telemetry pattern

IoT ingestion usually needs device identity, event timestamps, and tolerance for late or malformed readings.

In [ ]:
iot_source_path = "/Volumes/main/demo/raw/iot_sensor_events"
iot_checkpoint_path = "/Volumes/main/demo/checkpoints/iot_sensor_events"
iot_bronze_table = "main.demo.iot_sensor_bronze"
iot_gold_table = "main.demo.iot_temperature_5m_gold"

In [ ]:
iot_sensor_schema = StructType([
    StructField("device_id", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("sensor_type", StringType(), True),
    StructField("reading_value", DoubleType(), True),
    StructField("reading_ts", StringType(), True),
    StructField("unit", StringType(), True)
])

iot_bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", iot_checkpoint_path + "/schema")
    .schema(iot_sensor_schema)
    .load(iot_source_path)
    .withColumn("reading_ts", F.to_timestamp("reading_ts"))
    .withColumn("bronze_ingest_ts", F.current_timestamp())
)

iot_clean_stream_df = (
    iot_bronze_stream_df
    .filter(F.col("device_id").isNotNull())
    .filter(F.col("reading_ts").isNotNull())
    .filter(F.col("reading_value").isNotNull())
    .withWatermark("reading_ts", "10 minutes")
)

iot_temperature_agg_df = (
    iot_clean_stream_df
    .filter(F.col("sensor_type") == "temperature")
    .groupBy(F.window("reading_ts", "5 minutes"), "site_id", "device_id")
    .agg(F.avg("reading_value").alias("avg_temperature"))
)

iot_temperature_agg_df

## File landing with Auto Loader

This is a standard enterprise pattern for partner files, ERP extracts, and other incremental file drops.

In [ ]:
partner_source_path = "/Volumes/main/demo/raw/partner_orders"
partner_checkpoint_path = "/Volumes/main/demo/checkpoints/partner_orders"
partner_bronze_table = "main.demo.partner_orders_bronze"

partner_bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", partner_checkpoint_path + "/schema")
    .option("header", "true")
    .load(partner_source_path)
    .withColumn("source_file", F.input_file_name())
    .withColumn("bronze_ingest_ts", F.current_timestamp())
)

partner_bronze_stream_df

## API landing pattern

A common approach is to fetch API responses in Python, keep the raw payload in bronze, and flatten the response later in silver.

In [ ]:
api_bronze_table = "main.demo.crm_api_bronze"

api_payload_rows = [
    (
        "crm_contacts",
        "2026-04-05T06:00:00Z",
        "https://api.example.com/v1/contacts?updated_since=2026-04-05T00:00:00Z",
        "{\"contacts\":[{\"contact_id\":\"C100\",\"status\":\"active\"},{\"contact_id\":\"C101\",\"status\":\"inactive\"}]}"
    )
]

api_bronze_df = spark.createDataFrame(
    api_payload_rows,
    ["source_name", "request_ts_raw", "request_url", "raw_payload"]
)

api_bronze_df = (
    api_bronze_df
    .withColumn("request_ts", F.to_timestamp("request_ts_raw"))
    .drop("request_ts_raw")
    .withColumn("bronze_ingest_ts", F.current_timestamp())
)

api_bronze_df.write.format("delta").mode("overwrite").saveAsTable(api_bronze_table)
display(spark.table(api_bronze_table))

## Practical production notes

- Keep Kafka, IoT, file, and API ingestion in bronze first before applying business rules.
- Always capture source metadata such as offsets, file names, request timestamps, or device IDs.
- Add checkpoints and schema tracking for streaming and Auto Loader pipelines.
- Quarantine malformed or invalid records instead of losing them silently.
- Use silver and gold layers for standardization, quality checks, and business outputs.